<a href="https://colab.research.google.com/github/Harsh-Raghuvanshi/ActiveDomainAdaptation/blob/main/ResearchModel2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Installing Libraries (if not already installed)
!pip install torchvision numpy scikit-learn matplotlib

import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances_argmin_min


In [ ]:
# Loading MNIST and USPS Datasets
# Defining transformations for MNIST and USPS

transform = transforms.Compose([
    transforms.Resize((28, 28)),  # Ensure same size
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # Normalize to [-1, 1]
])

# Load MNIST (source domain)
mnist = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
mnist_test = datasets.MNIST(root='./data', train=False, transform=transform, download=True)

# Load USPS (target domain)
usps = datasets.USPS(root='./data', train=True, transform=transform, download=True)
usps_test = datasets.USPS(root='./data', train=False, transform=transform, download=True)

# DataLoaders
mnist_loader = DataLoader(mnist, batch_size=64, shuffle=True)
usps_loader = DataLoader(usps, batch_size=64, shuffle=True)


100%|██████████| 182M/182M [00:10<00:00, 17.7MB/s]


100%|██████████| 64.3M/64.3M [00:02<00:00, 21.5MB/s]


In [ ]:
transform_svhn = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  # Convert RGB to grayscale
    transforms.Resize((28, 28)),  # Ensure same size
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # Normalize to [-1, 1]
])

# Reload SVHN with the updated transform
svhn = datasets.SVHN(root='./data', split='train', transform=transform_svhn, download=True)
svhn_test = datasets.SVHN(root='./data', split='test', transform=transform_svhn, download=True)

# DataLoaders
svhn_loader = DataLoader(svhn, batch_size=64, shuffle=True)
svhn_test_loader = DataLoader(svhn_test, batch_size=64)


Using downloaded and verified file: ./data/train_32x32.mat
Using downloaded and verified file: ./data/test_32x32.mat


In [ ]:
# Cross Checking availability of data and deciding budget as 10% of target train set size
print("The size of mnist train and test is ", len(mnist), len(mnist_test))
print("The size of usps train and test is ", len(usps), len(usps_test))
print("The size of svhn train and test is ", len(svhn), len(svhn_test))
budget = int(0.1 * len(usps))

The size of mnist train and test is  60000 10000
The size of usps train and test is  7291 2007
The size of svhn train and test is  73257 26032


In [ ]:
# Defining the Simple Convultional neural network Model

import torch.nn as nn
import torch.optim as optim

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.fc = nn.Sequential(
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN().to(device)


In [ ]:
# Training on MNIST (Source Domain)

def train_source_domain(model, loader, epochs=5):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(loader)}")

train_source_domain(model, mnist_loader)
# Saving the base model after training for ressting it for various approaches
torch.save(model.state_dict(), '/content/models/base_cnn_model')




Epoch 1/5, Loss: 0.15389415387139677
Epoch 2/5, Loss: 0.044677917608807086
Epoch 3/5, Loss: 0.02959280218391035
Epoch 4/5, Loss: 0.021832328887135936
Epoch 5/5, Loss: 0.01666432194644725


In [ ]:
def resetting_base_model(model_path='/content/models/base_cnn_model'):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = SimpleCNN().to(device)
    try:
        model.load_state_dict(torch.load(model_path))
        print(f"Model successfully loaded from {model_path}.")
    except FileNotFoundError:
        print(f"Error: Model checkpoint not found at {model_path}")
        return None
    except Exception as e:
        print(f"Error while resetting model: {e}")
        return None

    model.train()  # Set to training mode for fine-tuning
    return model


In [ ]:
# Method Module [ will contain all necessary function for various active domain adaptation techniques]


# Compute Average Uncertainty
def compute_average_uncertainty(model, target_loader):
    model.eval()
    uncertainties = []

    with torch.no_grad():
        for images, _ in target_loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            entropy = -torch.sum(probs * torch.log(probs + 1e-5), dim=1)
            uncertainties.extend(entropy.cpu().numpy())

    return np.mean(uncertainties)


# Fine-tuning function
def fine_tune_model(model, loader, epochs=3):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0001)

    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
        print(f"Fine-tuning Epoch {epoch + 1}/{epochs}, Loss: {running_loss / len(loader)}")


# Uncertainty Sampling
def uncertainty_sampling(model, target_loader, n_samples=100):
    model.eval()
    uncertainties = []
    all_indices = []

    with torch.no_grad():
        for i, (images, _) in enumerate(target_loader):
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            entropy = -torch.sum(probs * torch.log(probs + 1e-5), dim=1)
            uncertainties.extend(entropy.cpu().numpy())
            all_indices.extend(range(i * target_loader.batch_size, (i + 1) * target_loader.batch_size))

    # Get indices of top uncertain samples
    uncertain_indices = np.argsort(uncertainties)[-n_samples:]
    return uncertain_indices


# Diversity Sampling
def diversity_sampling(target_data, n_samples=100):
    data = np.array([img.numpy().flatten() for img, _ in target_data])
    kmeans = KMeans(n_clusters=n_samples).fit(data)
    cluster_centers = kmeans.cluster_centers_
    indices, _ = pairwise_distances_argmin_min(cluster_centers, data)
    return indices


# Evaluating model
def evaluate_model(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f"Accuracy: {100 * correct / total:.2f}%")

In [ ]:

# Active Domain Adaptation using Uncertainity sampling with Fixed Budget

def active_domain_adaptation_with_uncertainity_sampling(
    model, target_loader, target_data, budget, n_iterations=5, batch_size=100
):
    labeled_samples = 0  # Track the total labeled samples
    total_samples = len(target_data)
    selected_indices = set()  # To avoid duplicate selections

    for iteration in range(n_iterations):
        if labeled_samples >= budget:
            print(f"Budget exhausted after {labeled_samples} samples.")
            break

        remaining_budget = budget - labeled_samples
        current_batch = min(batch_size, remaining_budget)  # Adjust batch size if close to budget

        print(f"Iteration {iteration + 1}: Sampling with weights - Uncertainty")

        # Perform Uncertainty Sampling
        uncertain_indices = uncertainty_sampling(model, target_loader, n_samples=current_batch)
        uncertain_indices = [idx for idx in uncertain_indices if idx not in selected_indices]


        # Combine Uncertainty and Diversity with Dynamic Weights
        combined_indices = (
            uncertain_indices[: int(current_batch)]
        )

        combined_indices = list(set(combined_indices))  # Remove duplicates
        selected_indices.update(combined_indices)

        # Update labeled samples
        labeled_samples += len(combined_indices)
        print(f"Iteration {iteration + 1}: Selected {len(combined_indices)} samples. Total labeled: {labeled_samples}/{budget}")

        # Fine-tune model on selected samples
        selected_data = Subset(target_data, combined_indices)
        selected_loader = DataLoader(selected_data, batch_size=32, shuffle=True)
        fine_tune_model(model, selected_loader)


    print("Active Domain Adaptation with Uncertainity Sampling completed.")



In [ ]:
#Active Domain adaptation using Diversity sampling with fixed Budget

def active_domain_adaptation_with_diversity_sampling(
    model, target_loader, target_data, budget, n_iterations=5, batch_size=100):
    labeled_samples = 0  # Track the total labeled samples
    total_samples = len(target_data)
    selected_indices = set()  # To avoid duplicate selections
    for iteration in range(n_iterations):
        if labeled_samples >= budget:
            print(f"Budget exhausted after {labeled_samples} samples.")
            break
        remaining_budget = budget - labeled_samples
        current_batch = min(batch_size, remaining_budget)  # Adjust batch size if close to budget
        print(f"Iteration {iteration + 1}: Sampling with weights - Diversity")
        diverse_indices = diversity_sampling(target_data, n_samples=current_batch)
        diverse_indices = [idx for idx in diverse_indices if idx not in selected_indices]
        combined_indices = (
            diverse_indices[: int(current_batch)]
        )
        combined_indices = list(set(combined_indices))  # Remove duplicates
        selected_indices.update(combined_indices)
        labeled_samples += len(combined_indices)
        print(f"Iteration {iteration + 1}: Selected {len(combined_indices)} samples. Total labeled: {labeled_samples}/{budget}")
        selected_data = Subset(target_data, combined_indices)
        selected_loader = DataLoader(selected_data, batch_size=32, shuffle=True)
        fine_tune_model(model, selected_loader)
    print("Active Domain Adaptation with Diversity Sampling completed.")



In [ ]:
# Adaptive Active Domain Adaptation using Hybrid approach with Fixed Budget

def active_domain_adaptation_with_fixed_weighting(
    model, target_loader, target_data, budget, n_iterations=5, batch_size=100
):
    labeled_samples = 0  # Track the total labeled samples
    total_samples = len(target_data)
    selected_indices = set()  # To avoid duplicate selections

    for iteration in range(n_iterations):
        if labeled_samples >= budget:
            print(f"Budget exhausted after {labeled_samples} samples.")
            break

        remaining_budget = budget - labeled_samples
        current_batch = min(batch_size, remaining_budget)  # Adjust batch size if close to budget

        # Adjust weights based on uncertainty reduction
        if n_iterations//2 < iteration:
          weight_uncertainty = 1
          weight_diversity = 0
        else:
          weight_uncertainty = 0
          weight_diversity = 1

        print(f"Iteration {iteration + 1}: Sampling with weights - Uncertainty: {weight_uncertainty}, Diversity: {weight_diversity}")

        # Perform Uncertainty Sampling
        uncertain_indices = uncertainty_sampling(model, target_loader, n_samples=current_batch)
        uncertain_indices = [idx for idx in uncertain_indices if idx not in selected_indices]

        # Perform Diversity Sampling
        diverse_indices = diversity_sampling(target_data, n_samples=current_batch)
        diverse_indices = [idx for idx in diverse_indices if idx not in selected_indices]

        # Combine Uncertainty and Diversity with Dynamic Weights
        combined_indices = (
            uncertain_indices[: int(current_batch * weight_uncertainty)]
            + diverse_indices[: int(current_batch * weight_diversity)]
        )

        combined_indices = list(set(combined_indices))  # Remove duplicates
        selected_indices.update(combined_indices)

        # Update labeled samples
        labeled_samples += len(combined_indices)
        print(f"Iteration {iteration + 1}: Selected {len(combined_indices)} samples. Total labeled: {labeled_samples}/{budget}")

        # Fine-tune model on selected samples
        selected_data = Subset(target_data, combined_indices)
        selected_loader = DataLoader(selected_data, batch_size=32, shuffle=True)
        fine_tune_model(model, selected_loader)

    print("Active Domain Adaptation with Dynamic Weighting completed.")




In [ ]:
# Adaptive Active Domain Adaptation with Dynamic Budget Allocation

def active_domain_adaptation_with_dynamic_weighting(
    model, target_loader, target_data, budget, n_iterations=5, batch_size=100
):
    labeled_samples = 0  # Track the total labeled samples
    total_samples = len(target_data)
    selected_indices = set()  # To avoid duplicate selections

    weight_uncertainty = 0.9  # Start with high weight on uncertainty sampling
    weight_diversity = 0.1

    for iteration in range(n_iterations):
        if labeled_samples >= budget:
            print(f"Budget exhausted after {labeled_samples} samples.")
            break

        remaining_budget = budget - labeled_samples
        current_batch = min(batch_size, remaining_budget)  # Adjust batch size if close to budget

        print(f"Iteration {iteration + 1}: Sampling with weights - Uncertainty: {weight_uncertainty}, Diversity: {weight_diversity}")

        # Perform Uncertainty Sampling
        uncertain_indices = uncertainty_sampling(model, target_loader, n_samples=current_batch)
        uncertain_indices = [idx for idx in uncertain_indices if idx not in selected_indices]

        # Perform Diversity Sampling
        diverse_indices = diversity_sampling(target_data, n_samples=current_batch)
        diverse_indices = [idx for idx in diverse_indices if idx not in selected_indices]

        # Combine Uncertainty and Diversity with Dynamic Weights
        combined_indices = (
            uncertain_indices[: int(current_batch * weight_uncertainty)]
            + diverse_indices[: int(current_batch * weight_diversity)]
        )

        combined_indices = list(set(combined_indices))  # Remove duplicates
        selected_indices.update(combined_indices)

        # Update labeled samples
        labeled_samples += len(combined_indices)
        print(f"Iteration {iteration + 1}: Selected {len(combined_indices)} samples. Total labeled: {labeled_samples}/{budget}")

        # Fine-tune model on selected samples
        selected_data = Subset(target_data, combined_indices)
        selected_loader = DataLoader(selected_data, batch_size=32, shuffle=True)
        fine_tune_model(model, selected_loader)

        # Adjust weights based on uncertainty reduction
        avg_uncertainty = compute_average_uncertainty(model, target_loader)
        if avg_uncertainty < 0.5:  # Example threshold for reducing uncertainty weight
            weight_uncertainty = max(0.1, weight_uncertainty - 0.15)
            weight_diversity = min(0.9, weight_diversity + 0.15)

    print("Active Domain Adaptation with Dynamic Weighting completed.")



In [ ]:
model=resetting_base_model()
print("The accuracy for Simple Model is on SVHN : ")
evaluate_model(model, DataLoader(svhn_test, batch_size=64))
print()

<ipython-input-10-08abd7a43186>:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path))


Model successfully loaded from /content/models/base_cnn_model.
The accuracy for Simple Model is on SVHN : 
Accuracy: 26.94%



In [ ]:
model=resetting_base_model()
active_domain_adaptation_with_dynamic_weighting(model,svhn_loader, svhn, budget=1000)
torch.save(model.state_dict(), '/content/models/dynamic_active_cnn_model_svhn')
print("The accuracy for dynamic weighted active domain adaptation using hybrid approach is : ")
evaluate_model(model, DataLoader(svhn_test, batch_size=64))
print()

<ipython-input-10-08abd7a43186>:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path))


Model successfully loaded from /content/models/base_cnn_model.
Iteration 1: Sampling with weights - Uncertainty: 0.9, Diversity: 0.1


KeyboardInterrupt: 

In [ ]:


model=resetting_base_model()
print("The accuracy for Simple Model is : ")
evaluate_model(model, DataLoader(usps_test, batch_size=64))
print()

model=resetting_base_model()
active_domain_adaptation_with_dynamic_weighting(model, usps_loader, usps, budget=1000)
torch.save(model.state_dict(), '/content/models/dynamic_active_cnn_model')
print("The accuracy for dynamic weighted active domain adaptation using hybrid approach is : ")
evaluate_model(model, DataLoader(usps_test, batch_size=64))
print()

model=resetting_base_model()
active_domain_adaptation_with_fixed_weighting(model, usps_loader, usps, budget=1000)
torch.save(model.state_dict(), '/content/models/static_active_cnn_model')
print("The accuracy for static weighted active domain adaptation using hybrid approach is : ")
evaluate_model(model, DataLoader(usps_test, batch_size=64))

model=resetting_base_model()
active_domain_adaptation_with_uncertainity_sampling(model, usps_loader, usps, budget=1000)
torch.save(model.state_dict(), '/content/models/uncertainity_active_cnn_model')
print("The accuracy for uncertainity active domain adaptation is : ")
evaluate_model(model, DataLoader(usps_test, batch_size=64))
print()

model=resetting_base_model()
active_domain_adaptation_with_diversity_sampling(model, usps_loader, usps, budget=1000)
torch.save(model.state_dict(), '/content/models/diversity_active_cnn_model')
print("The accuracy for diversity active domain adaptation is : ")
evaluate_model(model, DataLoader(usps_test, batch_size=64))
print()





<ipython-input-10-08abd7a43186>:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path))


Model successfully loaded from /content/models/base_cnn_model.
The accuracy for Simple Model is : 
Accuracy: 74.24%

Model successfully loaded from /content/models/base_cnn_model.
Iteration 1: Sampling with weights - Uncertainty: 0.9, Diversity: 0.1
Iteration 1: Selected 100 samples. Total labeled: 100/1000
Fine-tuning Epoch 1/3, Loss: 0.6688852496445179
Fine-tuning Epoch 2/3, Loss: 0.7544940859079361
Fine-tuning Epoch 3/3, Loss: 0.7740293443202972
Iteration 2: Sampling with weights - Uncertainty: 0.9, Diversity: 0.1
Iteration 2: Selected 100 samples. Total labeled: 200/1000
Fine-tuning Epoch 1/3, Loss: 0.8758775219321251
Fine-tuning Epoch 2/3, Loss: 0.5654615350067616
Fine-tuning Epoch 3/3, Loss: 0.6244158446788788
Iteration 3: Sampling with weights - Uncertainty: 0.9, Diversity: 0.1
Iteration 3: Selected 100 samples. Total labeled: 300/1000
Fine-tuning Epoch 1/3, Loss: 0.3785221390426159
Fine-tuning Epoch 2/3, Loss: 0.37295862287282944
Fine-tuning Epoch 3/3, Loss: 0.3072642832994461


<ipython-input-10-08abd7a43186>:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path))


Iteration 1: Selected 100 samples. Total labeled: 100/1000
Fine-tuning Epoch 1/3, Loss: 0.45812585949897766
Fine-tuning Epoch 2/3, Loss: 0.421674195677042
Fine-tuning Epoch 3/3, Loss: 0.36626889184117317
Iteration 2: Sampling with weights - Uncertainty: 0, Diversity: 1
Iteration 2: Selected 84 samples. Total labeled: 184/1000
Fine-tuning Epoch 1/3, Loss: 0.42302299042542774
Fine-tuning Epoch 2/3, Loss: 0.3789088229338328
Fine-tuning Epoch 3/3, Loss: 0.35625231514374417
Iteration 3: Sampling with weights - Uncertainty: 0, Diversity: 1
Iteration 3: Selected 71 samples. Total labeled: 255/1000
Fine-tuning Epoch 1/3, Loss: 0.31995804980397224
Fine-tuning Epoch 2/3, Loss: 0.3341023127237956
Fine-tuning Epoch 3/3, Loss: 0.4874042719602585
Iteration 4: Sampling with weights - Uncertainty: 1, Diversity: 0
Iteration 4: Selected 96 samples. Total labeled: 351/1000
Fine-tuning Epoch 1/3, Loss: 0.3421334425608317
Fine-tuning Epoch 2/3, Loss: 0.30917895833651227
Fine-tuning Epoch 3/3, Loss: 0.28181

<ipython-input-10-08abd7a43186>:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path))


Iteration 1: Selected 100 samples. Total labeled: 100/1000
Fine-tuning Epoch 1/3, Loss: 0.7304377108812332
Fine-tuning Epoch 2/3, Loss: 0.5227824449539185
Fine-tuning Epoch 3/3, Loss: 0.5094925686717033
Iteration 2: Sampling with weights - Uncertainty
Iteration 2: Selected 100 samples. Total labeled: 200/1000
Fine-tuning Epoch 1/3, Loss: 0.3722917176783085
Fine-tuning Epoch 2/3, Loss: 0.32583509013056755
Fine-tuning Epoch 3/3, Loss: 0.2799781057983637
Iteration 3: Sampling with weights - Uncertainty
Iteration 3: Selected 96 samples. Total labeled: 296/1000
Fine-tuning Epoch 1/3, Loss: 0.5846453905105591
Fine-tuning Epoch 2/3, Loss: 0.544532577196757
Fine-tuning Epoch 3/3, Loss: 0.5104574163754781
Iteration 4: Sampling with weights - Uncertainty
Iteration 4: Selected 96 samples. Total labeled: 392/1000
Fine-tuning Epoch 1/3, Loss: 0.39562422037124634
Fine-tuning Epoch 2/3, Loss: 0.36336763203144073
Fine-tuning Epoch 3/3, Loss: 0.3400430778662364
Iteration 5: Sampling with weights - Unce